# MSPT IMRE — full run có lưu data và tự resume

Notebook này tạo hai **Private Kaggle Datasets**: một dataset giữ `data.tar.gz` vĩnh viễn và một dataset giữ checkpoint sau từng task. Khi phiên 12 giờ kết thúc, chạy lại notebook; nó đọc `LAST_COMPLETED_TASK` và tiếp tục task kế tiếp.

Trước lần chạy đầu: bật GPU + Internet, attach dataset chứa repo `ContinueMKGC`, rồi thêm hai Kaggle Secrets `KAGGLE_USERNAME` và `KAGGLE_KEY`.

In [1]:
%%bash
# ===== Ô 2 =====

set -e
unset PYTHONPATH

CONDA_DIR=/kaggle/working/miniconda
ENV_DIR=/kaggle/working/imre-paper-env
READY="$ENV_DIR/.paper_ready"

# Cài Miniconda nếu chưa có
if [ ! -x "$CONDA_DIR/bin/conda" ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh \
        -O /kaggle/working/miniconda.sh

    bash /kaggle/working/miniconda.sh \
        -b \
        -p "$CONDA_DIR"
fi

# Accept ToS
"$CONDA_DIR/bin/conda" tos accept \
    --override-channels \
    --channel https://repo.anaconda.com/pkgs/main >/dev/null || true

"$CONDA_DIR/bin/conda" tos accept \
    --override-channels \
    --channel https://repo.anaconda.com/pkgs/r >/dev/null || true

# Python 3.9 riêng
if [ ! -x "$ENV_DIR/bin/python" ]; then
    "$CONDA_DIR/bin/conda" create \
        -y \
        -p "$ENV_DIR" \
        python=3.9 pip
fi

PY="$ENV_DIR/bin/python"

# Chỉ cài packages một lần trong session
if [ ! -f "$READY" ]; then
    "$PY" -m pip install -q "pip==23.3.2"

    "$PY" -m pip install -q \
        "numpy==1.21.5" \
        "Pillow==9.4.0" \
        "scikit-learn==1.2.0" \
        "tqdm==4.64.1" \
        "filelock==3.6.0" \
        "requests==2.28.1"

    "$PY" -m pip install -q \
        "torch==1.11.0+cu113" \
        "torchvision==0.12.0+cu113" \
        --extra-index-url https://download.pytorch.org/whl/cu113

    "$PY" -m pip install -q \
        "huggingface-hub==0.0.19" \
        "tokenizers==0.10.3" \
        "transformers==4.11.3"

    touch "$READY"
fi

# Dọn file tải/caches - không xóa env
rm -f /kaggle/working/miniconda.sh
rm -rf ~/.cache/pip
"$CONDA_DIR/bin/conda" clean -a -y >/dev/null 2>&1 || true

echo "===== PAPER ENV ====="
"$PY" - <<'PY'
import sys, torch, torchvision, transformers, numpy

print("Python       :", sys.version.split()[0])
print("Torch        :", torch.__version__)
print("Torchvision  :", torchvision.__version__)
print("Transformers :", transformers.__version__)
print("NumPy        :", numpy.__version__)
print("CUDA runtime :", torch.version.cuda)
print("CUDA usable  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
PY

PREFIX=/kaggle/working/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /kaggle/working/imre-paper-env

  added / updated specs:
    - pip
    - python=3.9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.9.25  |       h06a4308_0         107 KB
    expat-2.8.5                |       h7354ed3_1          24 KB
    libexpat-2.8.5             |       h7354ed3_1         129 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    openssl-3.5.9              |       h1b28b03_0         5.7 MB
    pip-26.0.1          



==> WARNING: A newer version of conda exists. <==
    current version: 26.7.1
    latest version: 26.9.1

Please update conda by running

    $ conda self update


WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



In [2]:
# ===== Ô 3 =====

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

from kaggle_secrets import UserSecretsClient


os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("PYTHONPATH", None)


# =====================================================
# CẤU HÌNH CHÍNH
# =====================================================

TASK_CONFIGS = [7]
EPOCHS = 10
BATCH_SIZE = 2

DATASET_NAME = "mspt-imre-official-data"

CHECKPOINT_DATASET_NAME = (
    "mspt-imre-checkpoints-sep7-wandb-rerun-v1"
)

# False: chạy mới hoàn toàn
# True: đọc checkpoint để resume
RESUME_FROM_CHECKPOINT = False


# =====================================================
# W&B CONFIG
# =====================================================

WANDB_PROJECT = "mspt-imre-reproduction"
WANDB_RUN_SUFFIX = "v1"


# =====================================================
# Kaggle Secrets
# =====================================================

secrets = UserSecretsClient()

KAGGLE_USERNAME = secrets.get_secret(
    "KAGGLE_USERNAME"
)

KAGGLE_KEY = secrets.get_secret(
    "KAGGLE_KEY"
)

WANDB_API_KEY = secrets.get_secret(
    "WANDB_API_KEY"
)


os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_MODE"] = "online"


# =====================================================
# Dataset IDs
# =====================================================

DATASET_ID = (
    f"{KAGGLE_USERNAME}/{DATASET_NAME}"
)

CHECKPOINT_DATASET_ID = (
    f"{KAGGLE_USERNAME}/{CHECKPOINT_DATASET_NAME}"
)


# =====================================================
# Paper environment
# =====================================================

WORKING = Path(
    "/kaggle/working"
)

PAPER_PYTHON = (
    WORKING
    / "imre-paper-env"
    / "bin"
    / "python"
)

assert PAPER_PYTHON.exists(), (
    f"Không tìm thấy paper env: {PAPER_PYTHON}"
)

print(
    "Paper Python:",
    PAPER_PYTHON
)


# =====================================================
# Cài W&B vào paper environment
# =====================================================

subprocess.run(
    [
        str(PAPER_PYTHON),
        "-m",
        "pip",
        "install",
        "-q",
        "wandb==0.17.9",
    ],
    check=True,
)

subprocess.run(
    [
        str(PAPER_PYTHON),
        "-c",
        "import wandb; print('W&B:', wandb.__version__)",
    ],
    check=True,
)


# =====================================================
# Copy source repository
# =====================================================

sources = list(
    Path("/kaggle/input").rglob("MRE/run.py")
)

if not sources:
    raise FileNotFoundError(
        "Attach Kaggle Dataset chứa repo ContinueMKGC "
        "trước khi Run All"
    )

SOURCE_REPO = (
    sources[0].parent.parent
)

REPO = (
    WORKING / "ContinueMKGC"
)

if REPO.exists():
    shutil.rmtree(REPO)


shutil.copytree(
    SOURCE_REPO,
    REPO,
    ignore=shutil.ignore_patterns(
        ".git",
        "__pycache__",
        "*.pyc",
    ),
)

MRE = REPO / "MRE"

print(
    "Repo:",
    SOURCE_REPO,
    "->",
    REPO
)

print(
    "W&B project:",
    WANDB_PROJECT
)

print(
    "W&B suffix:",
    WANDB_RUN_SUFFIX
)

print(
    "Resume from checkpoint:",
    RESUME_FROM_CHECKPOINT
)

Paper Python: /kaggle/working/imre-paper-env/bin/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.3/534.3 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.0 MB/s eta 0:00:00
W&B: 0.17.9
Repo: /kaggle/input/notebooks/lhongmh/mspt1/ContinueMKGC -> /kaggle/working/ContinueMKGC
W&B project: mspt-imre-reproduction
W&B suffix: v1
Resume from checkpoint: False


In [3]:
# ===== Ô 4 =====
# Compatibility patch + W&B patch


# =====================================================
# Patch run.py cho Transformers 4.11.3
# =====================================================

run_py = (
    MRE / "run.py"
)

text = run_py.read_text(
    encoding="utf-8"
)

text = text.replace(
    ", use_safetensors=False",
    "",
)

run_py.write_text(
    text,
    encoding="utf-8"
)

print(
    "Removed use_safetensors=False"
)


# =====================================================
# Patch train.py cho PyTorch 1.11
# =====================================================

train_py = (
    MRE / "modules" / "train.py"
)

text = train_py.read_text(
    encoding="utf-8"
)

text = text.replace(
    ", weights_only=False",
    "",
)

text = text.replace(
    "torch.set_rng_state(state[\"torch_rng\"])",
    "torch.set_rng_state(state[\"torch_rng\"].cpu())",
)

text = text.replace(
    "torch.cuda.set_rng_state_all(state[\"cuda_rng\"])",
    (
        "torch.cuda.set_rng_state_all("
        "[x.cpu() for x in state[\"cuda_rng\"]]"
        ")"
    ),
)


# Xóa checkpoint task trước sau khi resume thành công
if "import os" not in text:

    text = text.replace(
        "import torch",
        "import os\nimport torch",
        1,
    )


old_log = (
    'self.logger.info("Resumed after task %d from %s", '
    'task_id + 1, path)'
)

new_log = old_log + """

        # Xóa checkpoint cũ sau khi resume thành công
        old_checkpoint_files = [
            self.args.save_path
            + f"/model_{task_id}.pth",

            self.args.save_path
            + f"/share_{task_id}.pth",

            self.args.save_path
            + f"/resume_task_{task_id}.pth",
        ]

        for old_path in old_checkpoint_files:

            try:
                os.remove(old_path)

                self.logger.info(
                    "Removed local checkpoint: %s",
                    old_path,
                )

            except FileNotFoundError:
                pass
"""


if (
    old_log in text
    and "old_checkpoint_files" not in text
):

    text = text.replace(
        old_log,
        new_log,
        1,
    )


train_py.write_text(
    text,
    encoding="utf-8"
)

print(
    "Patched PyTorch compatibility"
)


# =====================================================
# W&B patch cho run.py
# =====================================================

run_py = (
    MRE / "run.py"
)

text = run_py.read_text(
    encoding="utf-8"
)


if "# === WANDB PATCH ===" not in text:

    text = text.replace(
        "from modules.train import BertTrainer\n",
        """from modules.train import BertTrainer

# === WANDB PATCH ===
try:
    import wandb
except Exception:
    wandb = None
""",
        1,
    )


    text = text.replace(
        "    parser = argparse.ArgumentParser()\n",
        """    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--wandb_project",
        default=None,
        type=str,
    )

    parser.add_argument(
        "--wandb_run_name",
        default=None,
        type=str,
    )

    parser.add_argument(
        "--wandb_run_id",
        default=None,
        type=str,
    )
""",
        1,
    )


    text = text.replace(
        "    args = parser.parse_args()\n",
        """    args = parser.parse_args()

    wandb_run = None

    if (
        wandb is not None
        and args.wandb_project
        and os.environ.get("WANDB_API_KEY")
    ):

        wandb_run = wandb.init(
            project=args.wandb_project,
            name=args.wandb_run_name,
            id=args.wandb_run_id,
            resume="allow",

            config={
                "model_name": args.model_name,
                "dataset_name": args.dataset_name,
                "task_number": args.task_number,
                "num_epochs": args.num_epochs,
                "batch_size": args.batch_size,
                "lr": args.lr,
                "seed": args.seed,
                "buffer_size": args.buffer_size,
                "do_replay": args.do_replay,
                "do_random": args.do_random,
                "do_balance": args.do_balance,
            },

            tags=[
                "MSPT",
                "IMRE",
                args.notes,
            ],
        )

        print(
            "W&B run:",
            wandb_run.url
        )
""",
        1,
    )


    text = text.replace(
        """        trainer.train()
        torch.cuda.empty_cache()
""",
        """        trainer.train()

        if wandb_run is not None:

            task_label = (
                args.start_task + 1
            )

            wandb.log(
                {
                    "final/task": task_label,
                    "final/best_dev_f1": (
                        trainer.best_dev_metric
                    ),
                    "final/best_test_f1": (
                        trainer.best_test_metric
                    ),
                    "final/best_dev_epoch": (
                        trainer.best_dev_epoch
                        if trainer.best_dev_epoch
                        is not None
                        else -1
                    ),
                    "final/best_test_epoch": (
                        trainer.best_test_epoch
                        if trainer.best_test_epoch
                        is not None
                        else -1
                    ),
                }
            )

            wandb.summary[
                "last_completed_task"
            ] = task_label

            wandb.finish()

        torch.cuda.empty_cache()
""",
        1,
    )


    run_py.write_text(
        text,
        encoding="utf-8"
    )

    print(
        "Patched run.py for W&B"
    )

else:

    print(
        "run.py already has W&B patch"
    )


# =====================================================
# W&B patch cho train.py
# Chỉ log dev theo epoch
# =====================================================

train_py = (
    MRE / "modules" / "train.py"
)

text = train_py.read_text(
    encoding="utf-8"
)


if "# === WANDB LOG PATCH ===" not in text:

    text = text.replace(
        "import gc\n",
        """import gc

# === WANDB LOG PATCH ===
try:
    import wandb
except Exception:
    wandb = None
""",
        1,
    )


    metric_line = (
        "                acc, micro_f1 = "
        "round(result['acc']*100, 4), "
        "round(result['micro_f1']*100, 4)\n"
    )


    dev_log = """                if (
                    wandb is not None
                    and wandb.run is not None
                ):

                    wandb.log(
                        {
                            "dev/acc": acc,
                            "dev/f1": micro_f1,
                            "task": task_id + 1,
                            "epoch": epoch,
                        }
                    )
"""


    if metric_line not in text:

        raise RuntimeError(
            "Không tìm thấy dòng metric trong train.py"
        )


    # Chỉ log metric dev theo epoch
    text = text.replace(
        metric_line,
        metric_line + dev_log,
        1,
    )


    train_py.write_text(
        text,
        encoding="utf-8"
    )

    print(
        "Patched train.py for W&B dev metrics"
    )

else:

    print(
        "train.py already has W&B patch"
    )


print(
    "Ô 4 hoàn tất"
)

Removed use_safetensors=False
Patched PyTorch compatibility
Patched run.py for W&B
Patched train.py for W&B dev metrics
Ô 4 hoàn tất


In [4]:
# ===== Ô 5 =====
def run(cmd, cwd=None, check=True):
    print('$', ' '.join(map(str, cmd)))
    return subprocess.run(
        list(map(str, cmd)),
        cwd=cwd,
        check=check,
        text=True
    )


def dataset_exists(dataset_id):
    result = subprocess.run(
        ['kaggle', 'datasets', 'files', dataset_id],
        capture_output=True,
        text=True
    )
    return result.returncode == 0


def md5sum(path):
    h = hashlib.md5()

    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)

    return h.hexdigest()


print(run(['nvidia-smi'], check=True))

$ nvidia-smi
Mon Oct  5 09:20:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------------

In [5]:
# ===== Ô 7 =====
DATA_SRC = next(
    Path('/kaggle/input').rglob('mspt-imre-official-data/data')
)

DATA_DST = MRE / 'data'

print("Official data:", DATA_SRC)
print("Repo data    :", DATA_DST)

# Check phần continual split phải có sẵn trong repo
for name in [
    'txt/sep5',
    'txt/sep7',
    'txt/sep10',
]:
    assert (DATA_DST / name).exists(), \
        f'Repo thiếu {name}'

# Các phần nặng lấy từ official dataset bằng symlink
for name in [
    'img_org',
    'img_vg',
    'img_detect',
]:
    src = DATA_SRC / name
    dst = DATA_DST / name

    assert src.exists(), \
        f'Official dataset thiếu {name}'

    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)

    dst.symlink_to(src, target_is_directory=True)

# Copy/link các file phụ trong txt nếu repo chưa có
for name in [
    'mre_train_dict.pth',
    'mre_dev_dict.pth',
    'mre_test_dict.pth',
]:
    src = DATA_SRC / 'txt' / name
    dst = DATA_DST / 'txt' / name

    assert src.exists(), \
        f'Official dataset thiếu txt/{name}'

    if not dst.exists():
        dst.symlink_to(src)

# Relation mapping
src = DATA_SRC / 'ours_rel2id.json'
dst = DATA_DST / 'ours_rel2id.json'

if not dst.exists():
    dst.symlink_to(src)

# Check cuối
required = [
    'img_org',
    'img_vg',
    'img_detect',
    'txt/sep5',
    'txt/sep7',
    'txt/sep10',
    'txt/mre_train_dict.pth',
    'txt/mre_dev_dict.pth',
    'txt/mre_test_dict.pth',
    'ours_rel2id.json',
]

for name in required:
    print(
        f'{name:25s}',
        (DATA_DST / name).exists()
    )

    assert (DATA_DST / name).exists(), \
        f'Thiếu {name}'

print("\nDATA READY")

Official data: /kaggle/input/datasets/nguyenmanhha2502/mspt-imre-official-data/data
Repo data    : /kaggle/working/ContinueMKGC/MRE/data
img_org                   True
img_vg                    True
img_detect                True
txt/sep5                  True
txt/sep7                  True
txt/sep10                 True
txt/mre_train_dict.pth    True
txt/mre_dev_dict.pth      True
txt/mre_test_dict.pth     True
ours_rel2id.json          True

DATA READY


In [6]:
# ===== Ô 8 =====
from huggingface_hub import snapshot_download

MODEL_ROOT = WORKING / "paper-models"

CLIP_LOCAL = MODEL_ROOT / "clip-vit-base-patch32"
BERT_LOCAL = MODEL_ROOT / "bert-base-uncased"

MODEL_ROOT.mkdir(parents=True, exist_ok=True)

clip_required = [
    "pytorch_model.bin",
    "config.json",
    "preprocessor_config.json",
    "vocab.json",
    "merges.txt",
]

bert_required = [
    "pytorch_model.bin",
    "config.json",
    "vocab.txt",
]

if not all((CLIP_LOCAL / x).exists() for x in clip_required):
    print("Downloading CLIP...")

    snapshot_download(
        repo_id="openai/clip-vit-base-patch32",
        local_dir=str(CLIP_LOCAL),
        allow_patterns=["*.json", "*.txt", "*.bin"],
    )
else:
    print("CLIP already ready")

if not all((BERT_LOCAL / x).exists() for x in bert_required):
    print("Downloading BERT...")

    snapshot_download(
        repo_id="bert-base-uncased",
        local_dir=str(BERT_LOCAL),
        allow_patterns=["*.json", "*.txt", "*.bin"],
    )
else:
    print("BERT already ready")

assert all(
    (CLIP_LOCAL / x).exists()
    for x in clip_required
)

assert all(
    (BERT_LOCAL / x).exists()
    for x in bert_required
)

print("CLIP:", CLIP_LOCAL)
print("BERT:", BERT_LOCAL)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

CLIP: /kaggle/working/paper-models/clip-vit-base-patch32
BERT: /kaggle/working/paper-models/bert-base-uncased


In [7]:
# ===== Ô 10 =====
# Train mới hoặc resume checkpoint + W&B


import json
import shutil
import subprocess
from pathlib import Path


# =====================================================
# Cấu hình
# =====================================================

assert len(TASK_CONFIGS) == 1, (
    "Mỗi session chỉ chạy một cấu hình."
)

assert "WANDB_PROJECT" in globals(), (
    "Thiếu WANDB_PROJECT. Hãy chạy Ô 3 trước."
)

assert "WANDB_RUN_SUFFIX" in globals(), (
    "Thiếu WANDB_RUN_SUFFIX. Hãy chạy Ô 3 trước."
)

assert "RESUME_FROM_CHECKPOINT" in globals(), (
    "Thiếu RESUME_FROM_CHECKPOINT. Hãy chạy Ô 3 trước."
)


TASK_COUNT = TASK_CONFIGS[0]
NOTES = f"sep{TASK_COUNT}"

WANDB_RUN_NAME = (
    f"IMRE-{NOTES}-full"
)

WANDB_RUN_ID = (
    f"imre-{NOTES}-{WANDB_RUN_SUFFIX}"
)


print(
    "W&B project:",
    WANDB_PROJECT
)

print(
    "W&B run:",
    WANDB_RUN_NAME
)

print(
    "W&B run ID:",
    WANDB_RUN_ID
)

print(
    "Resume mode:",
    RESUME_FROM_CHECKPOINT
)


# =====================================================
# Thư mục checkpoint riêng cho run này
# =====================================================

STATE_PACKAGE = (
    WORKING
    / f"imre-checkpoint-package-"
      f"{NOTES}-{WANDB_RUN_SUFFIX}"
)

BERT_DIR = (
    STATE_PACKAGE / "bert"
)

RUN_DIR = (
    BERT_DIR
    / f"MRE_4_1e-05_{NOTES}"
)

MARKER = (
    RUN_DIR / "LAST_COMPLETED_TASK"
)


# =====================================================
# Dọn file zip tạm
# =====================================================

def cleanup_upload_archives():

    for path in WORKING.glob("*.zip"):

        if path.is_file():

            print(
                "Removing temporary archive:",
                path
            )

            path.unlink()


    if STATE_PACKAGE.exists():

        for path in STATE_PACKAGE.rglob("*.zip"):

            if path.is_file():

                print(
                    "Removing temporary archive:",
                    path
                )

                path.unlink()


# =====================================================
# Kiểm tra checkpoint
# =====================================================

def checkpoint_is_ready():

    if not MARKER.exists():
        return False

    return True


# =====================================================
# Tìm checkpoint đúng sep
# =====================================================

def find_checkpoint_zip():

    expected_folder = (
        f"MRE_4_1e-05_{NOTES}"
    )

    candidates = []

    for path in Path("/kaggle/input").rglob("*.zip"):

        print(
            "Checking checkpoint archive:",
            path
        )

        result = subprocess.run(
            [
                "unzip",
                "-l",
                str(path),
            ],
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            continue

        if expected_folder in result.stdout:

            candidates.append(path)


    if not candidates:
        return None

    return candidates[0]


# =====================================================
# Fresh run hoặc resume
# =====================================================

cleanup_upload_archives()


if not RESUME_FROM_CHECKPOINT:

    # Chỉ xóa package riêng của run hiện tại
    if STATE_PACKAGE.exists():

        print(
            "Fresh run: xóa state package cũ:",
            STATE_PACKAGE
        )

        shutil.rmtree(STATE_PACKAGE)

else:

    print(
        "Resume mode: cho phép đọc checkpoint."
    )


STATE_PACKAGE.mkdir(
    parents=True,
    exist_ok=True
)

BERT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =====================================================
# Khôi phục checkpoint nếu bật resume
# =====================================================

if (
    RESUME_FROM_CHECKPOINT
    and not checkpoint_is_ready()
):

    checkpoint_zip = find_checkpoint_zip()

    if checkpoint_zip is None:

        print(
            f"Không tìm thấy checkpoint {NOTES}."
        )

        print(
            f"Bắt đầu {NOTES} từ task 1."
        )

    else:

        print(
            f"Tìm thấy checkpoint đúng {NOTES}:",
            checkpoint_zip
        )

        print(
            "Đang giải nén checkpoint..."
        )

        run(
            [
                "unzip",
                "-q",
                "-o",
                str(checkpoint_zip),
                "-d",
                str(STATE_PACKAGE),
            ]
        )

        cleanup_upload_archives()


elif not RESUME_FROM_CHECKPOINT:

    print(
        f"Fresh run: bỏ qua mọi checkpoint cũ của {NOTES}."
    )

    print(
        f"{NOTES} sẽ bắt đầu từ task 1."
    )


# =====================================================
# Chuẩn hóa thư mục checkpoint
# =====================================================

expected_folder = (
    f"MRE_4_1e-05_{NOTES}"
)


for src in STATE_PACKAGE.glob(
    "MRE_4_1e-05_*"
):

    if not src.is_dir():
        continue


    # Xóa checkpoint của sep khác
    if src.name != expected_folder:

        print(
            "Removing checkpoint of another sep:",
            src
        )

        shutil.rmtree(src)
        continue


    dst = BERT_DIR / src.name

    if not dst.exists():

        shutil.move(
            str(src),
            str(dst)
        )

        print(
            "Moved:",
            src,
            "->",
            dst
        )


RUN_DIR = (
    BERT_DIR / expected_folder
)

MARKER = (
    RUN_DIR / "LAST_COMPLETED_TASK"
)


# =====================================================
# Xác định task bắt đầu
# =====================================================

if (
    RESUME_FROM_CHECKPOINT
    and MARKER.exists()
):

    completed_task = int(
        MARKER.read_text(
            encoding="utf-8"
        ).strip()
    )

    start_task = (
        completed_task + 1
    )

    print(
        f"{NOTES} đã hoàn thành đến task "
        f"{completed_task + 1}/{TASK_COUNT}"
    )

    if start_task < TASK_COUNT:

        print(
            f"Sẽ tiếp tục từ task "
            f"{start_task + 1}/{TASK_COUNT}"
        )

    else:

        print(
            f"{NOTES} đã hoàn thành đủ "
            f"{TASK_COUNT}/{TASK_COUNT}"
        )

else:

    start_task = 0

    print(
        f"Bắt đầu mới từ task "
        f"1/{TASK_COUNT}"
    )


# =====================================================
# Metadata Kaggle Dataset
# =====================================================

state_meta = {
    "title": CHECKPOINT_DATASET_NAME,
    "id": CHECKPOINT_DATASET_ID,
    "licenses": [
        {
            "name": "other"
        }
    ]
}


(
    STATE_PACKAGE / "dataset-metadata.json"
).write_text(
    json.dumps(state_meta),
    encoding="utf-8"
)


# =====================================================
# Upload checkpoint
# =====================================================

def publish_checkpoint(message):

    cleanup_upload_archives()

    print(
        "Uploading checkpoint:",
        message
    )


    if dataset_exists(
        CHECKPOINT_DATASET_ID
    ):

        run(
            [
                "kaggle",
                "datasets",
                "version",
                "-p",
                str(STATE_PACKAGE),
                "-m",
                message,
                "--dir-mode",
                "zip",
            ]
        )

    else:

        run(
            [
                "kaggle",
                "datasets",
                "create",
                "-p",
                str(STATE_PACKAGE),
                "--dir-mode",
                "zip",
            ]
        )


    cleanup_upload_archives()


# =====================================================
# Train
# =====================================================

MIN_FREE_GB = 7


for task_id in range(
    start_task,
    TASK_COUNT
):

    print(
        f"\n===== {NOTES}: "
        f"task {task_id + 1}/{TASK_COUNT} ====="
    )


    cmd = [
        str(PAPER_PYTHON),
        "-u",
        "run.py",

        "--model_name",
        "bert",

        "--dataset_name",
        "MRE",

        "--bert_name",
        str(BERT_LOCAL),

        "--vit_name",
        str(CLIP_LOCAL),

        "--num_epochs",
        str(EPOCHS),

        "--batch_size",
        str(BATCH_SIZE),

        "--local_files_only",

        "--lr",
        "0.00001",

        "--eval_begin_epoch",
        "1",

        "--seed",
        "1234",

        "--do_train",
        "--do_save",
        "--do_replay",
        "--do_random",

        "--max_seq",
        "80",

        "--prompt_len",
        "4",

        "--aux_size",
        "128",

        "--rcnn_size",
        "64",

        "--do_balance",

        "--alpha",
        "0.1",

        "--gamma",
        "1",

        "--key_lr",
        "0.000001",

        "--bias_lr",
        "0.00001",

        "--type_text",
        "width",

        "--type_vision",
        "width",

        "--task_number",
        str(TASK_COUNT),

        "--max_tasks",
        str(task_id + 1),

        "--start_task",
        str(task_id),

        "--notes",
        NOTES,

        # W&B
        "--wandb_project",
        WANDB_PROJECT,

        "--wandb_run_name",
        WANDB_RUN_NAME,

        "--wandb_run_id",
        WANDB_RUN_ID,

        # Checkpoint
        "--save_path",
        str(STATE_PACKAGE),
    ]


    # =================================================
    # Kiểm tra dung lượng
    # =================================================

    free_gb = (
        shutil.disk_usage(WORKING).free
        / (1024 ** 3)
    )

    print(
        f"Free disk before task: "
        f"{free_gb:.2f} GB"
    )


    if (
        task_id == 0
        and free_gb < MIN_FREE_GB
    ):

        raise RuntimeError(
            f"Chỉ còn {free_gb:.2f} GB. "
            "Không đủ dung lượng an toàn."
        )


    # =================================================
    # Kiểm tra checkpoint task trước
    # =================================================

    if task_id > 0:

        previous_task = (
            task_id - 1
        )

        required_files = [
            RUN_DIR
            / f"model_{previous_task}.pth",

            RUN_DIR
            / f"share_{previous_task}.pth",

            RUN_DIR
            / f"resume_task_{previous_task}.pth",
        ]

        missing = [
            str(path)
            for path in required_files
            if not path.exists()
        ]

        if missing:

            raise FileNotFoundError(
                "Thiếu checkpoint để resume:\n"
                + "\n".join(missing)
            )


    # =================================================
    # Chạy task
    # =================================================

    run(
        cmd,
        cwd=MRE
    )


    # =================================================
    # Chỉ giữ checkpoint task hiện tại
    # =================================================

    for old in RUN_DIR.glob("*.pth"):

        if not old.name.endswith(
            f"_{task_id}.pth"
        ):

            old.unlink()


    # =================================================
    # Upload checkpoint
    # =================================================

    publish_checkpoint(
        f"{NOTES} completed task "
        f"{task_id + 1}/{TASK_COUNT}"
    )


    print(
        f"SAVED: {NOTES} task "
        f"{task_id + 1}/{TASK_COUNT}"
    )


# =====================================================
# Kết thúc
# =====================================================

if start_task >= TASK_COUNT:

    print(
        f"{NOTES} đã hoàn thành đủ "
        f"{TASK_COUNT}/{TASK_COUNT}"
    )

else:

    print(
        f"\nPASS: {NOTES} hoàn thành "
        f"{TASK_COUNT}/{TASK_COUNT}"
    )


print(
    "Local checkpoint:",
    RUN_DIR
)

print(
    "Remote checkpoint dataset:",
    CHECKPOINT_DATASET_ID
)

print(
    "W&B project:",
    WANDB_PROJECT
)

print(
    "W&B run:",
    WANDB_RUN_NAME
)

print(
    "W&B URL sẽ xuất hiện trong log của run.py."
)

W&B project: mspt-imre-reproduction
W&B run: IMRE-sep7-full
W&B run ID: imre-sep7-v1
Resume mode: False
Fresh run: bỏ qua mọi checkpoint cũ của sep7.
sep7 sẽ bắt đầu từ task 1.
Bắt đầu mới từ task 1/7

===== sep7: task 1/7 =====
Free disk before task: 13.46 GB
$ /kaggle/working/imre-paper-env/bin/python -u run.py --model_name bert --dataset_name MRE --bert_name /kaggle/working/paper-models/bert-base-uncased --vit_name /kaggle/working/paper-models/clip-vit-base-patch32 --num_epochs 10 --batch_size 2 --local_files_only --lr 0.00001 --eval_begin_epoch 1 --seed 1234 --do_train --do_save --do_replay --do_random --max_seq 80 --prompt_len 4 --aux_size 128 --rcnn_size 64 --do_balance --alpha 0.1 --gamma 1 --key_lr 0.000001 --bias_lr 0.00001 --type_text width --type_vision width --task_number 7 --max_tasks 1 --start_task 0 --notes sep7 --wandb_project mspt-imre-reproduction --wandb_run_name IMRE-sep7-full --wandb_run_id imre-sep7-v1 --save_path /kaggle/working/imre-checkpoint-package-sep7-v1
./

wandb: Currently logged in as: lhmha2910 (lhm_ha). Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.17.9
wandb: Run data is saved locally in /kaggle/working/ContinueMKGC/MRE/wandb/run-20261005_092200-imre-sep7-v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run IMRE-sep7-full
wandb: ⭐️ View project at https://wandb.ai/lhm_ha/mspt-imre-reproduction
wandb: 🚀 View run at https://wandb.ai/lhm_ha/mspt-imre-reproduction/runs/imre-sep7-v1
ftfy or spacy is not installed using BERT BasicTokenizer instead of ftfy.


W&B run: https://wandb.ai/lhm_ha/mspt-imre-reproduction/runs/imre-sep7-v1
地址: data/txt/sep7/ours_train_
Namespace(wandb_project='mspt-imre-reproduction', wandb_run_name='IMRE-sep7-full', wandb_run_id='imre-sep7-v1', model_name='bert', vit_name='/kaggle/working/paper-models/clip-vit-base-patch32', dataset_name='MRE', bert_name='/kaggle/working/paper-models/bert-base-uncased', num_epochs=10, device='cuda', batch_size=2, lr=1e-05, warmup_ratio=0.01, eval_begin_epoch=1, seed=1234, load_path=None, save_path='/kaggle/working/imre-checkpoint-package-sep7-v1/bert/MRE_4_1e-05_sep7', write_path=None, notes='sep7', do_train=True, do_test=False, do_predict=False, prompt_len=4, max_seq=80, aux_size=128, rcnn_size=64, do_balance=True, alpha=0.1, gamma=1.0, key_lr=1e-06, bias_lr=1e-05, type_text='width', type_vision='width', do_text_modify=False, do_task_finetune=False, do_save=True, task_id=4, task_number=7, max_tasks=1, start_task=0, local_files_only=True, do_replay=True, do_random=True, buffer_siz

10/05/2026 09:22:03 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_train_0.txt
10/05/2026 09:22:04 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_train_1.txt
10/05/2026 09:22:04 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_train_2.txt
10/05/2026 09:22:04 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_train_3.txt
10/05/2026 09:22:04 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_train_4.txt
10/05/2026 09:22:04 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_train_5.txt
10/05/2026 09:22:05 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_train_6.txt
10/05/2026 09:22:05 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_val_0.txt
10/05/2026 09:22:05 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_val_1.txt
10/05/2026 09:22:05 - INFO - processor.dataset -   Loading data from data/txt/sep7/ours_val_2.txt
10/05/

CLIPVisionConfig {
  "attention_dropout": 0.0,
  "dropout": 0.0,
  "hidden_act": "quick_gelu",
  "hidden_size": 768,
  "image_size": 224,
  "initializer_factor": 1.0,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "model_type": "clip_vision_model",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "patch_size": 32,
  "transformers_version": "4.11.3"
}

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.11.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size"

...

CLIPVisionConfig {
  "attention_dropout": 0.0,
  "dropout": 0.0,
  "hidden_act": "quick_gelu",
  "hidden_size": 768,
  "image_size": 224,
  "initializer_factor": 1.0,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "model_type": "clip_vision_model",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "patch_size": 32,
  "transformers_version": "4.11.3"
}

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.11.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size"

10/05/2026 15:59:19 - INFO - __main__ -   Resumed after task 6 from /kaggle/working/imre-checkpoint-package-sep7-v1/bert/MRE_4_1e-05_sep7/resume_task_5.pth
10/05/2026 15:59:19 - INFO - __main__ -   Removed local checkpoint: /kaggle/working/imre-checkpoint-package-sep7-v1/bert/MRE_4_1e-05_sep7/model_5.pth
10/05/2026 15:59:19 - INFO - __main__ -   Removed local checkpoint: /kaggle/working/imre-checkpoint-package-sep7-v1/bert/MRE_4_1e-05_sep7/share_5.pth
10/05/2026 15:59:19 - INFO - __main__ -   Removed local checkpoint: /kaggle/working/imre-checkpoint-package-sep7-v1/bert/MRE_4_1e-05_sep7/resume_task_5.pth
10/05/2026 15:59:19 - INFO - __main__ -   ***** Running training *****
10/05/2026 15:59:19 - INFO - __main__ -     Num instance = 1296
10/05/2026 15:59:19 - INFO - __main__ -     Num epoch = 10
10/05/2026 15:59:19 - INFO - __main__ -     Batch size = 2
10/05/2026 15:59:19 - INFO - __main__ -     Learning rate = 1e-05
10/05/2026 15:59:19 - INFO - __main__ -     Evaluate begin = 1
Task 7

Uploading checkpoint: sep7 completed task 7/7
$ kaggle datasets version -p /kaggle/working/imre-checkpoint-package-sep7-v1 -m sep7 completed task 7/7 --dir-mode zip
Starting upload for file bert.zip


100%|██████████| 4.49G/4.49G [00:43<00:00, 111MB/s]


Upload successful: bert.zip (4GB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/hlhongm/mspt-imre-checkpoints-sep7-wandb-rerun-v1
SAVED: sep7 task 7/7

PASS: sep7 hoàn thành 7/7
Local checkpoint: /kaggle/working/imre-checkpoint-package-sep7-v1/bert/MRE_4_1e-05_sep7
Remote checkpoint dataset: hlhongm/mspt-imre-checkpoints-sep7-wandb-rerun-v1
W&B project: mspt-imre-reproduction
W&B run: IMRE-sep7-full
W&B URL sẽ xuất hiện trong log của run.py.
